# Socceraction

## Libraries and Constants

In [1]:
# Import libraries
import pandas as pd
import socceraction.spadl as spadl

from pathlib import Path
from socceraction.data.statsbomb import StatsBombLoader
from tqdm import tqdm

In [2]:
# Ignore warnings
import warnings

warnings.filterwarnings(action="ignore", message="Inferred xy_fidelity_version=2.", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [3]:
# Define the data directory
DATA_DIR = Path("../data")
SPADL_DIR = Path("data")

SPADL_H5 = SPADL_DIR / "spadl-statsbomb.h5"

## StatsBomb to SPADL

In [4]:
# Define the StatsBomb API as local
statsbomb_data_dir = str(DATA_DIR)
SBL = StatsBombLoader(getter="local", root=statsbomb_data_dir)

In [5]:
# Load the competitions data
competitions_df = SBL.competitions()
competitions_df

,season_id,competition_id,competition_name,country_name,competition_gender,season_name
0,281,9,1. Bundesliga,Germany,male,2023/2024
1,27,9,1. Bundesliga,Germany,male,2015/2016
2,107,1267,African Cup of Nations,Africa,male,2023
3,4,16,Champions League,Europe,male,2018/2019
4,1,16,Champions League,Europe,male,2017/2018
...,...,...,...,...,...,...
69,43,55,UEFA Euro,Europe,male,2020
70,75,35,UEFA Europa League,Europe,male,1988/1989
71,106,53,UEFA Women's Euro,Europe,female,2022
72,107,72,Women's World Cup,International,female,2023


In [6]:
# Get the number of games for each competition and season pair
competition_season_pairs = list(competitions_df[["competition_id", "season_id"]].itertuples(index=False))

number_of_games = []
for competition_id, season_id in competition_season_pairs:
    row_games_df = SBL.games(competition_id=competition_id, season_id=season_id)
    number_of_games.append(len(row_games_df))

competitions_with_games = competitions_df.copy()
competitions_with_games["number_of_games"] = number_of_games
print(competitions_with_games.to_string())

    season_id  competition_id         competition_name               country_name competition_gender season_name  number_of_games
0         281               9            1. Bundesliga                    Germany               male   2023/2024               34
1          27               9            1. Bundesliga                    Germany               male   2015/2016              306
2         107            1267   African Cup of Nations                     Africa               male        2023               52
3           4              16         Champions League                     Europe               male   2018/2019                1
4           1              16         Champions League                     Europe               male   2017/2018                1
5           2              16         Champions League                     Europe               male   2016/2017                1
6          27              16         Champions League                     Europe         

In [7]:
# Filter the competitions to keep only the Top 5 Leagues of the 2015/2016 season

# top_5_leagues = ["Premier League", "La Liga", "Serie A", "1. Bundesliga", "Ligue 1"]
# selected_competitions = competitions[
#     competitions["competition_name"].isin(top_5_leagues) & (competitions["season_name"] == "2015/2016")
# ]

# We will only use the Premier League for efficiency
selected_competitions_df = competitions_df[
    (competitions_df["competition_name"] == "Premier League") & (competitions_df["season_name"] == "2015/2016")
]
selected_competitions_df

,season_id,competition_id,competition_name,country_name,competition_gender,season_name
64,27,2,Premier League,England,male,2015/2016


In [8]:
# Load the games data for the selected competitions
competition_season_pairs = list(selected_competitions_df[["competition_id", "season_id"]].itertuples(index=False))

games_df = pd.DataFrame()
for competition_id, season_id in competition_season_pairs:
    league_games_df = SBL.games(competition_id=competition_id, season_id=season_id)
    games_df = pd.concat([games_df, league_games_df], ignore_index=True)
games_df

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,3754058,27,2,Regular Season,20,2016-01-02 16:00:00,22,28,0,0,King Power Stadium,Andre Marriner
1,3754245,27,2,Regular Season,9,2015-10-17 16:00:00,27,41,1,0,The Hawthorns,Martin Atkinson
2,3754136,27,2,Regular Season,17,2015-12-19 18:30:00,37,59,1,1,St. James'' Park,Martin Atkinson
3,3754037,27,2,Regular Season,36,2016-04-30 16:00:00,29,28,2,1,Goodison Park,Neil Swarbrick
4,3754039,27,2,Regular Season,26,2016-02-13 16:00:00,31,23,1,2,Selhurst Park,Robert Madley
...,...,...,...,...,...,...,...,...,...,...,...,...
375,3754020,27,2,Regular Season,2,2015-08-17 21:00:00,24,28,1,0,Anfield,Craig Pawson
376,3754267,27,2,Regular Season,2,2015-08-15 16:00:00,23,27,0,0,Vicarage Road,Paul Tierney
377,3754141,27,2,Regular Season,1,2015-08-09 14:30:00,1,40,0,2,Emirates Stadium,Martin Atkinson
378,3754128,27,2,Regular Season,1,2015-08-08 16:00:00,28,59,0,1,Vitality Stadium,Mark Clattenburg


## SPADL to HDF5

In [9]:
# Load the maches data and convert them to SPADL format
game_home_team_pair = list(games_df[["game_id", "home_team_id"]].itertuples(index=False))
teams, players = [], []
actions = {}

for game_id, home_team_id in tqdm(game_home_team_pair, desc="Loading game data"):
    # Load teams, players, and events for each game
    teams.append(SBL.teams(game_id=game_id))
    players.append(SBL.players(game_id=game_id))
    events = SBL.events(game_id=game_id)

    # Convert events to SPADL actions
    actions[game_id] = spadl.statsbomb.convert_to_actions(
        events, home_team_id=home_team_id, xy_fidelity_version=1, shot_fidelity_version=1
    )

teams_df = pd.concat(teams).drop_duplicates(subset="team_id")
players_df = pd.concat(players)

Loading game data: 100%|██████████| 380/380 [04:31<00:00,  1.40it/s]


In [10]:
# Save the data to an HDF5 file
with pd.HDFStore(SPADL_H5) as spadl_store:
    spadl_store["competitions"] = selected_competitions_df
    spadl_store["games"] = games_df
    spadl_store["teams"] = teams_df
    spadl_store["players"] = players_df[["player_id", "player_name", "nickname"]].drop_duplicates(subset="player_id")
    spadl_store["player_games"] = players_df[
        [
            "player_id",
            "game_id",
            "team_id",
            "is_starter",
            "starting_position_id",
            "starting_position_name",
            "minutes_played",
        ]
    ]
    for game_id in actions.keys():
        spadl_store[f"actions/game_{game_id}"] = actions[game_id]